In [0]:
#display the data from the table payroll
display(spark.read.table("plworkforce_catalog.001_bronze.payroll"))

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# Use silver schema
spark.sql("USE plworkforce_catalog.002_silver")

# Read Bronze
df = spark.table("plworkforce_catalog.001_bronze.payroll")

# Transform
df_clean = (df

    # Convert dates (your format is special)
    .withColumn("pay_period_start", F.to_timestamp("pay_period_start", "dd-MM-yyyy HH:mm"))
    .withColumn("pay_period_end", F.to_timestamp("pay_period_end", "dd-MM-yyyy HH:mm"))
    .withColumn("pay_date", F.to_timestamp("pay_date", "dd-MM-yyyy HH:mm"))

    # Cast numeric columns
    .withColumn("gross_salary", F.col("gross_salary").cast(DecimalType(18,2)))
    .withColumn("bonus", F.col("bonus").cast(DecimalType(18,2)))
    .withColumn("overtime_pay", F.col("overtime_pay").cast(DecimalType(18,2)))
    .withColumn("commission", F.col("commission").cast(DecimalType(18,2)))
    .withColumn("allowances", F.col("allowances").cast(DecimalType(18,2)))

    .withColumn("tax_deduction", F.col("tax_deduction").cast(DecimalType(18,2)))
    .withColumn("social_security", F.col("social_security").cast(DecimalType(18,2)))
    .withColumn("health_insurance", F.col("health_insurance").cast(DecimalType(18,2)))
    .withColumn("retirement_contribution", F.col("retirement_contribution").cast(DecimalType(18,2)))
    .withColumn("other_deductions", F.col("other_deductions").cast(DecimalType(18,2)))

    # 🔥 Recalculate net salary (important for marks)
    .withColumn("calculated_net_salary",
        F.col("gross_salary") +
        F.col("bonus") +
        F.col("overtime_pay") +
        F.col("commission") +
        F.col("allowances") -
        (
            F.col("tax_deduction") +
            F.col("social_security") +
            F.col("health_insurance") +
            F.col("retirement_contribution") +
            F.col("other_deductions")
        )
    )

    # Standardize text
    .withColumn("status", F.upper(F.col("status")))
    .withColumn("payment_method", F.initcap(F.col("payment_method")))

    # Derived columns (for Gold KPIs 🔥)
    .withColumn("year", F.year("pay_date"))
    .withColumn("month", F.month("pay_date"))
    .withColumn("year_month", F.date_format("pay_date", "yyyy-MM"))

    # Data quality
    .dropDuplicates(["payroll_id"])
    .filter(F.col("employee_id").isNotNull())
)

# Write to Silver
(df_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("payroll")
)

print("Silver payroll created")

#to display the data from the table payroll silver
display(spark.read.table("plworkforce_catalog.002_silver.payroll"))